# OntoBridge lakehouse build

Generated from alignment `run-20261021T084623-b7c8` for a Fabric Lakehouse (Delta); upload data/sources to Files/raw first.

## Finance

In [ ]:
%%sql
SET spark.sql.session.timeZone = UTC

In [ ]:
%%sql
CREATE SCHEMA IF NOT EXISTS bronze

In [ ]:
%%sql
CREATE SCHEMA IF NOT EXISTS silver

In [ ]:
%%sql
CREATE SCHEMA IF NOT EXISTS finance

In [ ]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW raw_finance_branch_registry USING csv OPTIONS (path 'Files/raw/finance/branch_registry.csv', header 'true')

In [ ]:
%%sql
DROP TABLE IF EXISTS bronze.finance_branch_registry

In [ ]:
%%sql
CREATE TABLE bronze.finance_branch_registry USING DELTA AS
SELECT * FROM raw_finance_branch_registry

In [ ]:
%%sql
DROP TABLE IF EXISTS silver.finance_branch_registry

In [ ]:
%%sql
CREATE TABLE silver.finance_branch_registry USING DELTA AS
SELECT
  CAST(`branch_id` AS STRING) AS branch_id,
  CAST(`branch_name` AS STRING) AS branch_name,
  CAST(`branch_city` AS STRING) AS city,
  CAST(`branch_state` AS STRING) AS state_code,
  CAST(`opened_on` AS DATE) AS branch_open_date,
  CAST(`manager_name` AS STRING) AS manager_name,
  'finance/branch_registry.csv' AS _source
FROM bronze.finance_branch_registry

In [ ]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW raw_finance_card_transactions USING json OPTIONS (path 'Files/raw/finance/card_transactions.json', multiLine 'true')

In [ ]:
%%sql
DROP TABLE IF EXISTS bronze.finance_card_transactions

In [ ]:
%%sql
CREATE TABLE bronze.finance_card_transactions USING DELTA AS
SELECT * FROM raw_finance_card_transactions

In [ ]:
%%sql
DROP TABLE IF EXISTS silver.finance_card_transactions

In [ ]:
%%sql
CREATE TABLE silver.finance_card_transactions USING DELTA AS
SELECT
  CAST(`txnRef` AS STRING) AS card_transaction_id,
  CAST(`card`.`last4` AS STRING) AS card_last_4,
  CAST(`card`.`holderId` AS STRING) AS customer_id,
  CAST(`auth_ts` AS TIMESTAMP) AS authorization_timestamp,
  CAST(`amount`.`value` AS DECIMAL(18,2)) AS transaction_amount,
  CAST(`amount`.`currency` AS STRING) AS currency_code,
  CAST(`merchant`.`name` AS STRING) AS merchant_name,
  CAST(`merchant`.`mcc` AS STRING) AS merchant_category_code,
  CAST(`merchant`.`city` AS STRING) AS city,
  CAST(`merchant`.`country` AS STRING) AS country_code,
  CAST(`isFraudFlag` AS BOOLEAN) AS fraud_flag,
  'finance/card_transactions.json' AS _source
FROM bronze.finance_card_transactions

In [ ]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW raw_finance_ledger_entries USING csv OPTIONS (path 'Files/raw/finance/ledger_entries.csv', header 'true')

In [ ]:
%%sql
DROP TABLE IF EXISTS bronze.finance_ledger_entries

In [ ]:
%%sql
CREATE TABLE bronze.finance_ledger_entries USING DELTA AS
SELECT * FROM raw_finance_ledger_entries

In [ ]:
%%sql
DROP TABLE IF EXISTS silver.finance_ledger_entries

In [ ]:
%%sql
CREATE TABLE silver.finance_ledger_entries USING DELTA AS
SELECT
  CAST(`je_id` AS STRING) AS journal_entry_id,
  CAST(`acct_no` AS STRING) AS account_number,
  CAST(`posting_date` AS DATE) AS posting_date,
  CAST(`debit_amt` AS DECIMAL(18,2)) AS debit_amount,
  CAST(`credit_amt` AS DECIMAL(18,2)) AS credit_amount,
  CAST(`cost_center` AS STRING) AS cost_center,
  CAST(`currency_cd` AS STRING) AS currency_code,
  'finance/ledger_entries.csv' AS _source
FROM bronze.finance_ledger_entries

In [ ]:
%%sql
DROP TABLE IF EXISTS finance.dim_account

In [ ]:
%%sql
CREATE TABLE finance.dim_account USING DELTA AS
SELECT xxhash64(account_number) AS account_key, account_number, max(cost_center) AS cost_center
FROM (
  SELECT account_number, cost_center FROM silver.finance_ledger_entries
) src
WHERE account_number IS NOT NULL
GROUP BY account_number
UNION ALL
SELECT -1, 'UNKNOWN', NULL

In [ ]:
%%sql
DROP TABLE IF EXISTS finance.dim_branch

In [ ]:
%%sql
CREATE TABLE finance.dim_branch USING DELTA AS
SELECT xxhash64(branch_id) AS branch_key, branch_id, max(branch_name) AS branch_name, max(manager_name) AS manager_name, max(city) AS city, max(state_code) AS state_code, max(branch_open_date) AS branch_open_date
FROM (
  SELECT branch_id, branch_name, manager_name, city, state_code, branch_open_date FROM silver.finance_branch_registry
) src
WHERE branch_id IS NOT NULL
GROUP BY branch_id
UNION ALL
SELECT -1, 'UNKNOWN', NULL, NULL, NULL, NULL, NULL

In [ ]:
%%sql
DROP TABLE IF EXISTS finance.dim_currency

In [ ]:
%%sql
CREATE TABLE finance.dim_currency USING DELTA AS
SELECT xxhash64(currency_code) AS currency_key, currency_code
FROM (
  SELECT currency_code FROM silver.finance_card_transactions
  UNION ALL
  SELECT currency_code FROM silver.finance_ledger_entries
) src
WHERE currency_code IS NOT NULL
GROUP BY currency_code
UNION ALL
SELECT -1, 'UNKNOWN'

In [ ]:
%%sql
DROP TABLE IF EXISTS finance.dim_customer

In [ ]:
%%sql
CREATE TABLE finance.dim_customer USING DELTA AS
SELECT xxhash64(customer_id) AS customer_key, customer_id
FROM (
  SELECT customer_id FROM silver.finance_card_transactions
) src
WHERE customer_id IS NOT NULL
GROUP BY customer_id
UNION ALL
SELECT -1, 'UNKNOWN'

In [ ]:
%%sql
DROP TABLE IF EXISTS finance.dim_merchant

In [ ]:
%%sql
CREATE TABLE finance.dim_merchant USING DELTA AS
SELECT xxhash64(merchant_name) AS merchant_key, merchant_name, max(merchant_category_code) AS merchant_category_code
FROM (
  SELECT merchant_name, merchant_category_code FROM silver.finance_card_transactions
) src
WHERE merchant_name IS NOT NULL
GROUP BY merchant_name
UNION ALL
SELECT -1, 'UNKNOWN', NULL

In [ ]:
%%sql
DROP TABLE IF EXISTS finance.dim_location

In [ ]:
%%sql
CREATE TABLE finance.dim_location USING DELTA AS
SELECT DISTINCT CASE WHEN coalesce(city, country_code) IS NULL THEN -1 ELSE xxhash64(coalesce(CAST(city AS STRING), ''), coalesce(CAST(country_code AS STRING), '')) END AS location_key, city, country_code
FROM (
  SELECT city, country_code FROM silver.finance_card_transactions
) src
WHERE coalesce(city, country_code) IS NOT NULL
UNION ALL
SELECT -1, NULL, NULL

In [ ]:
%%sql
DROP TABLE IF EXISTS finance.dim_date

In [ ]:
%%sql
CREATE TABLE finance.dim_date USING DELTA AS
SELECT CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key, d AS calendar_date, year(d) AS year,
  quarter(d) AS quarter, month(d) AS month, date_format(d, 'MMMM') AS month_name,
  dayofmonth(d) AS day_of_month, date_format(d, 'EEEE') AS day_name, dayofweek(d) IN (1, 7) AS is_weekend
FROM (SELECT explode(sequence(lo, hi, INTERVAL 1 DAY)) AS d
  FROM (SELECT min(d) AS lo, max(d) AS hi FROM (
    SELECT CAST(authorization_timestamp AS DATE) AS d FROM silver.finance_card_transactions
    UNION ALL
    SELECT CAST(posting_date AS DATE) AS d FROM silver.finance_ledger_entries
  ) dates) bounds)
UNION ALL
SELECT -1, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL

In [ ]:
%%sql
DROP TABLE IF EXISTS finance.fact_card_transactions

In [ ]:
%%sql
CREATE TABLE finance.fact_card_transactions USING DELTA AS
SELECT
  CASE WHEN currency_code IS NULL THEN -1 ELSE xxhash64(currency_code) END AS currency_key,
  CASE WHEN customer_id IS NULL THEN -1 ELSE xxhash64(customer_id) END AS customer_key,
  CASE WHEN merchant_name IS NULL THEN -1 ELSE xxhash64(merchant_name) END AS merchant_key,
  CASE WHEN coalesce(city, country_code) IS NULL THEN -1 ELSE xxhash64(coalesce(CAST(city AS STRING), ''), coalesce(CAST(country_code AS STRING), '')) END AS location_key,
  coalesce(CAST(date_format(authorization_timestamp, 'yyyyMMdd') AS INT), -1) AS authorization_date_key,
  card_transaction_id,
  card_last_4,
  authorization_timestamp,
  fraud_flag,
  transaction_amount,
  _source
FROM silver.finance_card_transactions

In [ ]:
%%sql
DROP TABLE IF EXISTS finance.fact_ledger_entries

In [ ]:
%%sql
CREATE TABLE finance.fact_ledger_entries USING DELTA AS
SELECT
  CASE WHEN account_number IS NULL THEN -1 ELSE xxhash64(account_number) END AS account_key,
  CASE WHEN currency_code IS NULL THEN -1 ELSE xxhash64(currency_code) END AS currency_key,
  coalesce(CAST(date_format(posting_date, 'yyyyMMdd') AS INT), -1) AS posting_date_key,
  journal_entry_id,
  debit_amount,
  credit_amount,
  _source
FROM silver.finance_ledger_entries

Validation for finance: every count must be 0.

In [ ]:
%%sql
SELECT 'unique finance.dim_account.account_key' AS check, (SELECT count(*) - count(DISTINCT account_key) FROM finance.dim_account) AS violations
UNION ALL
SELECT 'unique finance.dim_branch.branch_key' AS check, (SELECT count(*) - count(DISTINCT branch_key) FROM finance.dim_branch) AS violations
UNION ALL
SELECT 'unique finance.dim_currency.currency_key' AS check, (SELECT count(*) - count(DISTINCT currency_key) FROM finance.dim_currency) AS violations
UNION ALL
SELECT 'unique finance.dim_customer.customer_key' AS check, (SELECT count(*) - count(DISTINCT customer_key) FROM finance.dim_customer) AS violations
UNION ALL
SELECT 'unique finance.dim_merchant.merchant_key' AS check, (SELECT count(*) - count(DISTINCT merchant_key) FROM finance.dim_merchant) AS violations
UNION ALL
SELECT 'unique finance.dim_location.location_key' AS check, (SELECT count(*) - count(DISTINCT location_key) FROM finance.dim_location) AS violations
UNION ALL
SELECT 'unique finance.dim_date.date_key' AS check, (SELECT count(*) - count(DISTINCT date_key) FROM finance.dim_date) AS violations
UNION ALL
SELECT 'rows finance.fact_card_transactions' AS check, (SELECT (SELECT count(*) FROM finance.fact_card_transactions) - (SELECT count(*) FROM silver.finance_card_transactions)) AS violations
UNION ALL
SELECT 'orphans finance.fact_card_transactions.currency_key' AS check, (SELECT count(*) FROM finance.fact_card_transactions f LEFT ANTI JOIN finance.dim_currency d ON f.currency_key = d.currency_key) AS violations
UNION ALL
SELECT 'orphans finance.fact_card_transactions.customer_key' AS check, (SELECT count(*) FROM finance.fact_card_transactions f LEFT ANTI JOIN finance.dim_customer d ON f.customer_key = d.customer_key) AS violations
UNION ALL
SELECT 'orphans finance.fact_card_transactions.merchant_key' AS check, (SELECT count(*) FROM finance.fact_card_transactions f LEFT ANTI JOIN finance.dim_merchant d ON f.merchant_key = d.merchant_key) AS violations
UNION ALL
SELECT 'orphans finance.fact_card_transactions.location_key' AS check, (SELECT count(*) FROM finance.fact_card_transactions f LEFT ANTI JOIN finance.dim_location d ON f.location_key = d.location_key) AS violations
UNION ALL
SELECT 'orphans finance.fact_card_transactions.authorization_date_key' AS check, (SELECT count(*) FROM finance.fact_card_transactions f LEFT ANTI JOIN finance.dim_date d ON f.authorization_date_key = d.date_key) AS violations
UNION ALL
SELECT 'rows finance.fact_ledger_entries' AS check, (SELECT (SELECT count(*) FROM finance.fact_ledger_entries) - (SELECT count(*) FROM silver.finance_ledger_entries)) AS violations
UNION ALL
SELECT 'orphans finance.fact_ledger_entries.account_key' AS check, (SELECT count(*) FROM finance.fact_ledger_entries f LEFT ANTI JOIN finance.dim_account d ON f.account_key = d.account_key) AS violations
UNION ALL
SELECT 'orphans finance.fact_ledger_entries.currency_key' AS check, (SELECT count(*) FROM finance.fact_ledger_entries f LEFT ANTI JOIN finance.dim_currency d ON f.currency_key = d.currency_key) AS violations
UNION ALL
SELECT 'orphans finance.fact_ledger_entries.posting_date_key' AS check, (SELECT count(*) FROM finance.fact_ledger_entries f LEFT ANTI JOIN finance.dim_date d ON f.posting_date_key = d.date_key) AS violations

## Healthcare

In [ ]:
%%sql
SET spark.sql.session.timeZone = UTC

In [ ]:
%%sql
CREATE SCHEMA IF NOT EXISTS bronze

In [ ]:
%%sql
CREATE SCHEMA IF NOT EXISTS silver

In [ ]:
%%sql
CREATE SCHEMA IF NOT EXISTS healthcare

In [ ]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW raw_healthcare_ehr_encounters USING json OPTIONS (path 'Files/raw/healthcare/ehr_encounters.json', multiLine 'true')

In [ ]:
%%sql
DROP TABLE IF EXISTS bronze.healthcare_ehr_encounters

In [ ]:
%%sql
CREATE TABLE bronze.healthcare_ehr_encounters USING DELTA AS
SELECT * FROM raw_healthcare_ehr_encounters

In [ ]:
%%sql
DROP TABLE IF EXISTS silver.healthcare_ehr_encounters

In [ ]:
%%sql
CREATE TABLE silver.healthcare_ehr_encounters USING DELTA AS
SELECT
  CAST(`encounterId` AS STRING) AS encounter_id,
  CAST(`patient`.`mrn` AS STRING) AS patient_mrn,
  CAST(`patient`.`dob` AS DATE) AS patient_birth_date,
  CAST(`patient`.`sex` AS STRING) AS patient_sex,
  CAST(`admitDate` AS DATE) AS admit_date,
  CAST(`dischargeDate` AS DATE) AS discharge_date,
  CAST(`encounterType` AS STRING) AS encounter_type,
  CAST(`primaryDx` AS STRING) AS primary_diagnosis_code,
  CAST(`attendingNpi` AS STRING) AS provider_npi,
  CAST(`facility`.`code` AS STRING) AS facility_code,
  CAST(`facility`.`state` AS STRING) AS state_code,
  CAST(`totalCharges` AS DECIMAL(18,2)) AS total_charges,
  'healthcare/ehr_encounters.json' AS _source
FROM bronze.healthcare_ehr_encounters

In [ ]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW raw_healthcare_pharmacy_fills USING csv OPTIONS (path 'Files/raw/healthcare/pharmacy_fills.csv', header 'true')

In [ ]:
%%sql
DROP TABLE IF EXISTS bronze.healthcare_pharmacy_fills

In [ ]:
%%sql
CREATE TABLE bronze.healthcare_pharmacy_fills USING DELTA AS
SELECT * FROM raw_healthcare_pharmacy_fills

In [ ]:
%%sql
DROP TABLE IF EXISTS silver.healthcare_pharmacy_fills

In [ ]:
%%sql
CREATE TABLE silver.healthcare_pharmacy_fills USING DELTA AS
SELECT
  CAST(`rx_id` AS STRING) AS prescription_id,
  CAST(`patient_mrn` AS STRING) AS patient_mrn,
  CAST(`ndc_code` AS STRING) AS ndc_code,
  CAST(`fill_dt` AS DATE) AS fill_date,
  CAST(`days_supply` AS INT) AS days_supply,
  CAST(`qty_dispensed` AS DECIMAL(18,2)) AS quantity_dispensed,
  CAST(`copay_amt` AS DECIMAL(18,2)) AS copay_amount,
  CAST(`prescriber_npi` AS STRING) AS provider_npi,
  'healthcare/pharmacy_fills.csv' AS _source
FROM bronze.healthcare_pharmacy_fills

In [ ]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW raw_healthcare_provider_directory USING csv OPTIONS (path 'Files/raw/healthcare/provider_directory.csv', header 'true')

In [ ]:
%%sql
DROP TABLE IF EXISTS bronze.healthcare_provider_directory

In [ ]:
%%sql
CREATE TABLE bronze.healthcare_provider_directory USING DELTA AS
SELECT * FROM raw_healthcare_provider_directory

In [ ]:
%%sql
DROP TABLE IF EXISTS silver.healthcare_provider_directory

In [ ]:
%%sql
CREATE TABLE silver.healthcare_provider_directory USING DELTA AS
SELECT
  CAST(`npi` AS STRING) AS provider_npi,
  CAST(`provider_name` AS STRING) AS provider_name,
  CAST(`specialty` AS STRING) AS provider_specialty,
  CAST(`practice_zip` AS STRING) AS postal_code,
  CAST(`practice_state` AS STRING) AS state_code,
  CAST(`accepting_new` AS BOOLEAN) AS accepting_new_patients,
  'healthcare/provider_directory.csv' AS _source
FROM bronze.healthcare_provider_directory

In [ ]:
%%sql
DROP TABLE IF EXISTS healthcare.dim_facility

In [ ]:
%%sql
CREATE TABLE healthcare.dim_facility USING DELTA AS
SELECT xxhash64(facility_code) AS facility_key, facility_code
FROM (
  SELECT facility_code FROM silver.healthcare_ehr_encounters
) src
WHERE facility_code IS NOT NULL
GROUP BY facility_code
UNION ALL
SELECT -1, 'UNKNOWN'

In [ ]:
%%sql
DROP TABLE IF EXISTS healthcare.dim_patient

In [ ]:
%%sql
CREATE TABLE healthcare.dim_patient USING DELTA AS
SELECT xxhash64(patient_mrn) AS patient_key, patient_mrn, max(patient_birth_date) AS patient_birth_date, max(patient_sex) AS patient_sex
FROM (
  SELECT patient_mrn, patient_birth_date, patient_sex FROM silver.healthcare_ehr_encounters
  UNION ALL
  SELECT patient_mrn, CAST(NULL AS DATE) AS patient_birth_date, CAST(NULL AS STRING) AS patient_sex FROM silver.healthcare_pharmacy_fills
) src
WHERE patient_mrn IS NOT NULL
GROUP BY patient_mrn
UNION ALL
SELECT -1, 'UNKNOWN', NULL, NULL

In [ ]:
%%sql
DROP TABLE IF EXISTS healthcare.dim_provider

In [ ]:
%%sql
CREATE TABLE healthcare.dim_provider USING DELTA AS
SELECT xxhash64(provider_npi) AS provider_key, provider_npi, max(accepting_new_patients) AS accepting_new_patients, max(provider_name) AS provider_name, max(provider_specialty) AS provider_specialty, max(postal_code) AS postal_code, max(state_code) AS state_code
FROM (
  SELECT provider_npi, CAST(NULL AS BOOLEAN) AS accepting_new_patients, CAST(NULL AS STRING) AS provider_name, CAST(NULL AS STRING) AS provider_specialty, CAST(NULL AS STRING) AS postal_code, state_code FROM silver.healthcare_ehr_encounters
  UNION ALL
  SELECT provider_npi, CAST(NULL AS BOOLEAN) AS accepting_new_patients, CAST(NULL AS STRING) AS provider_name, CAST(NULL AS STRING) AS provider_specialty, CAST(NULL AS STRING) AS postal_code, CAST(NULL AS STRING) AS state_code FROM silver.healthcare_pharmacy_fills
  UNION ALL
  SELECT provider_npi, accepting_new_patients, provider_name, provider_specialty, postal_code, state_code FROM silver.healthcare_provider_directory
) src
WHERE provider_npi IS NOT NULL
GROUP BY provider_npi
UNION ALL
SELECT -1, 'UNKNOWN', NULL, NULL, NULL, NULL, NULL

In [ ]:
%%sql
DROP TABLE IF EXISTS healthcare.dim_location

In [ ]:
%%sql
CREATE TABLE healthcare.dim_location USING DELTA AS
SELECT DISTINCT CASE WHEN coalesce(state_code) IS NULL THEN -1 ELSE xxhash64(coalesce(CAST(state_code AS STRING), '')) END AS location_key, state_code
FROM (
  SELECT state_code FROM silver.healthcare_ehr_encounters
) src
WHERE coalesce(state_code) IS NOT NULL
UNION ALL
SELECT -1, NULL

In [ ]:
%%sql
DROP TABLE IF EXISTS healthcare.dim_date

In [ ]:
%%sql
CREATE TABLE healthcare.dim_date USING DELTA AS
SELECT CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key, d AS calendar_date, year(d) AS year,
  quarter(d) AS quarter, month(d) AS month, date_format(d, 'MMMM') AS month_name,
  dayofmonth(d) AS day_of_month, date_format(d, 'EEEE') AS day_name, dayofweek(d) IN (1, 7) AS is_weekend
FROM (SELECT explode(sequence(lo, hi, INTERVAL 1 DAY)) AS d
  FROM (SELECT min(d) AS lo, max(d) AS hi FROM (
    SELECT CAST(admit_date AS DATE) AS d FROM silver.healthcare_ehr_encounters
    UNION ALL
    SELECT CAST(discharge_date AS DATE) AS d FROM silver.healthcare_ehr_encounters
    UNION ALL
    SELECT CAST(fill_date AS DATE) AS d FROM silver.healthcare_pharmacy_fills
  ) dates) bounds)
UNION ALL
SELECT -1, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL

In [ ]:
%%sql
DROP TABLE IF EXISTS healthcare.fact_ehr_encounters

In [ ]:
%%sql
CREATE TABLE healthcare.fact_ehr_encounters USING DELTA AS
SELECT
  CASE WHEN facility_code IS NULL THEN -1 ELSE xxhash64(facility_code) END AS facility_key,
  CASE WHEN patient_mrn IS NULL THEN -1 ELSE xxhash64(patient_mrn) END AS patient_key,
  CASE WHEN provider_npi IS NULL THEN -1 ELSE xxhash64(provider_npi) END AS provider_key,
  CASE WHEN coalesce(state_code) IS NULL THEN -1 ELSE xxhash64(coalesce(CAST(state_code AS STRING), '')) END AS location_key,
  coalesce(CAST(date_format(admit_date, 'yyyyMMdd') AS INT), -1) AS admit_date_key,
  coalesce(CAST(date_format(discharge_date, 'yyyyMMdd') AS INT), -1) AS discharge_date_key,
  encounter_id,
  encounter_type,
  primary_diagnosis_code,
  total_charges,
  _source
FROM silver.healthcare_ehr_encounters

In [ ]:
%%sql
DROP TABLE IF EXISTS healthcare.fact_pharmacy_fills

In [ ]:
%%sql
CREATE TABLE healthcare.fact_pharmacy_fills USING DELTA AS
SELECT
  CASE WHEN patient_mrn IS NULL THEN -1 ELSE xxhash64(patient_mrn) END AS patient_key,
  CASE WHEN provider_npi IS NULL THEN -1 ELSE xxhash64(provider_npi) END AS provider_key,
  coalesce(CAST(date_format(fill_date, 'yyyyMMdd') AS INT), -1) AS fill_date_key,
  prescription_id,
  ndc_code,
  days_supply,
  quantity_dispensed,
  copay_amount,
  _source
FROM silver.healthcare_pharmacy_fills

Validation for healthcare: every count must be 0.

In [ ]:
%%sql
SELECT 'unique healthcare.dim_facility.facility_key' AS check, (SELECT count(*) - count(DISTINCT facility_key) FROM healthcare.dim_facility) AS violations
UNION ALL
SELECT 'unique healthcare.dim_patient.patient_key' AS check, (SELECT count(*) - count(DISTINCT patient_key) FROM healthcare.dim_patient) AS violations
UNION ALL
SELECT 'unique healthcare.dim_provider.provider_key' AS check, (SELECT count(*) - count(DISTINCT provider_key) FROM healthcare.dim_provider) AS violations
UNION ALL
SELECT 'unique healthcare.dim_location.location_key' AS check, (SELECT count(*) - count(DISTINCT location_key) FROM healthcare.dim_location) AS violations
UNION ALL
SELECT 'unique healthcare.dim_date.date_key' AS check, (SELECT count(*) - count(DISTINCT date_key) FROM healthcare.dim_date) AS violations
UNION ALL
SELECT 'rows healthcare.fact_ehr_encounters' AS check, (SELECT (SELECT count(*) FROM healthcare.fact_ehr_encounters) - (SELECT count(*) FROM silver.healthcare_ehr_encounters)) AS violations
UNION ALL
SELECT 'orphans healthcare.fact_ehr_encounters.facility_key' AS check, (SELECT count(*) FROM healthcare.fact_ehr_encounters f LEFT ANTI JOIN healthcare.dim_facility d ON f.facility_key = d.facility_key) AS violations
UNION ALL
SELECT 'orphans healthcare.fact_ehr_encounters.patient_key' AS check, (SELECT count(*) FROM healthcare.fact_ehr_encounters f LEFT ANTI JOIN healthcare.dim_patient d ON f.patient_key = d.patient_key) AS violations
UNION ALL
SELECT 'orphans healthcare.fact_ehr_encounters.provider_key' AS check, (SELECT count(*) FROM healthcare.fact_ehr_encounters f LEFT ANTI JOIN healthcare.dim_provider d ON f.provider_key = d.provider_key) AS violations
UNION ALL
SELECT 'orphans healthcare.fact_ehr_encounters.location_key' AS check, (SELECT count(*) FROM healthcare.fact_ehr_encounters f LEFT ANTI JOIN healthcare.dim_location d ON f.location_key = d.location_key) AS violations
UNION ALL
SELECT 'orphans healthcare.fact_ehr_encounters.admit_date_key' AS check, (SELECT count(*) FROM healthcare.fact_ehr_encounters f LEFT ANTI JOIN healthcare.dim_date d ON f.admit_date_key = d.date_key) AS violations
UNION ALL
SELECT 'orphans healthcare.fact_ehr_encounters.discharge_date_key' AS check, (SELECT count(*) FROM healthcare.fact_ehr_encounters f LEFT ANTI JOIN healthcare.dim_date d ON f.discharge_date_key = d.date_key) AS violations
UNION ALL
SELECT 'rows healthcare.fact_pharmacy_fills' AS check, (SELECT (SELECT count(*) FROM healthcare.fact_pharmacy_fills) - (SELECT count(*) FROM silver.healthcare_pharmacy_fills)) AS violations
UNION ALL
SELECT 'orphans healthcare.fact_pharmacy_fills.patient_key' AS check, (SELECT count(*) FROM healthcare.fact_pharmacy_fills f LEFT ANTI JOIN healthcare.dim_patient d ON f.patient_key = d.patient_key) AS violations
UNION ALL
SELECT 'orphans healthcare.fact_pharmacy_fills.provider_key' AS check, (SELECT count(*) FROM healthcare.fact_pharmacy_fills f LEFT ANTI JOIN healthcare.dim_provider d ON f.provider_key = d.provider_key) AS violations
UNION ALL
SELECT 'orphans healthcare.fact_pharmacy_fills.fill_date_key' AS check, (SELECT count(*) FROM healthcare.fact_pharmacy_fills f LEFT ANTI JOIN healthcare.dim_date d ON f.fill_date_key = d.date_key) AS violations

## Insurance

In [ ]:
%%sql
SET spark.sql.session.timeZone = UTC

In [ ]:
%%sql
CREATE SCHEMA IF NOT EXISTS bronze

In [ ]:
%%sql
CREATE SCHEMA IF NOT EXISTS silver

In [ ]:
%%sql
CREATE SCHEMA IF NOT EXISTS insurance

In [ ]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW raw_insurance_agents_crm USING csv OPTIONS (path 'Files/raw/insurance/agents_crm.csv', header 'true')

In [ ]:
%%sql
DROP TABLE IF EXISTS bronze.insurance_agents_crm

In [ ]:
%%sql
CREATE TABLE bronze.insurance_agents_crm USING DELTA AS
SELECT * FROM raw_insurance_agents_crm

In [ ]:
%%sql
DROP TABLE IF EXISTS silver.insurance_agents_crm

In [ ]:
%%sql
CREATE TABLE silver.insurance_agents_crm USING DELTA AS
SELECT
  CAST(`agent_code` AS STRING) AS agent_id,
  CAST(`agent_full_name` AS STRING) AS agent_name,
  CAST(`agency` AS STRING) AS agency_name,
  CAST(`region` AS STRING) AS sales_region,
  CAST(`hire_date` AS DATE) AS agent_hire_date,
  CAST(`email` AS STRING) AS agent_email,
  'insurance/agents_crm.csv' AS _source
FROM bronze.insurance_agents_crm

In [ ]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW raw_insurance_claims_events USING json OPTIONS (path 'Files/raw/insurance/claims_events.json', multiLine 'true')

In [ ]:
%%sql
DROP TABLE IF EXISTS bronze.insurance_claims_events

In [ ]:
%%sql
CREATE TABLE bronze.insurance_claims_events USING DELTA AS
SELECT * FROM raw_insurance_claims_events

In [ ]:
%%sql
DROP TABLE IF EXISTS silver.insurance_claims_events

In [ ]:
%%sql
CREATE TABLE silver.insurance_claims_events USING DELTA AS
SELECT
  CAST(`claimId` AS STRING) AS claim_id,
  CAST(`policy`.`number` AS STRING) AS policy_number,
  CAST(`lossDate` AS DATE) AS loss_date,
  CAST(`reportedAt` AS TIMESTAMP) AS claim_reported_timestamp,
  CAST(`claimant`.`customerRef` AS STRING) AS customer_id,
  CAST(`claimant`.`fullName` AS STRING) AS customer_name,
  CAST(`reserveAmount` AS DECIMAL(18,2)) AS reserve_amount,
  CAST(`paidAmount` AS DECIMAL(18,2)) AS paid_amount,
  CAST(`status` AS STRING) AS claim_status,
  CAST(`lossCause` AS STRING) AS loss_cause,
  CAST(`location`.`zip` AS STRING) AS postal_code,
  CAST(`location`.`state` AS STRING) AS state_code,
  'insurance/claims_events.json' AS _source
FROM bronze.insurance_claims_events

In [ ]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW raw_insurance_policy_admin USING csv OPTIONS (path 'Files/raw/insurance/policy_admin.csv', header 'true')

In [ ]:
%%sql
DROP TABLE IF EXISTS bronze.insurance_policy_admin

In [ ]:
%%sql
CREATE TABLE bronze.insurance_policy_admin USING DELTA AS
SELECT * FROM raw_insurance_policy_admin

In [ ]:
%%sql
DROP TABLE IF EXISTS silver.insurance_policy_admin

In [ ]:
%%sql
CREATE TABLE silver.insurance_policy_admin USING DELTA AS
SELECT
  CAST(`POL_NO` AS STRING) AS policy_number,
  CAST(`HOLDER_ID` AS STRING) AS customer_id,
  CAST(`HOLDER_NM` AS STRING) AS customer_name,
  CAST(`EFF_DT` AS DATE) AS policy_effective_date,
  CAST(`EXP_DT` AS DATE) AS policy_expiration_date,
  CAST(`PREM_AMT` AS DECIMAL(18,2)) AS premium_amount,
  CAST(`LOB_CD` AS STRING) AS line_of_business,
  CAST(`ST_CD` AS STRING) AS state_code,
  CAST(`AGT_ID` AS STRING) AS agent_id,
  'insurance/policy_admin.csv' AS _source
FROM bronze.insurance_policy_admin

In [ ]:
%%sql
DROP TABLE IF EXISTS insurance.dim_agent

In [ ]:
%%sql
CREATE TABLE insurance.dim_agent USING DELTA AS
SELECT xxhash64(agent_id) AS agent_key, agent_id, max(agency_name) AS agency_name, max(agent_email) AS agent_email, max(agent_name) AS agent_name, max(sales_region) AS sales_region, max(agent_hire_date) AS agent_hire_date
FROM (
  SELECT agent_id, agency_name, agent_email, agent_name, sales_region, agent_hire_date FROM silver.insurance_agents_crm
  UNION ALL
  SELECT agent_id, CAST(NULL AS STRING) AS agency_name, CAST(NULL AS STRING) AS agent_email, CAST(NULL AS STRING) AS agent_name, CAST(NULL AS STRING) AS sales_region, CAST(NULL AS DATE) AS agent_hire_date FROM silver.insurance_policy_admin
) src
WHERE agent_id IS NOT NULL
GROUP BY agent_id
UNION ALL
SELECT -1, 'UNKNOWN', NULL, NULL, NULL, NULL, NULL

In [ ]:
%%sql
DROP TABLE IF EXISTS insurance.dim_customer

In [ ]:
%%sql
CREATE TABLE insurance.dim_customer USING DELTA AS
SELECT xxhash64(customer_id) AS customer_key, customer_id, max(customer_name) AS customer_name
FROM (
  SELECT customer_id, customer_name FROM silver.insurance_claims_events
  UNION ALL
  SELECT customer_id, customer_name FROM silver.insurance_policy_admin
) src
WHERE customer_id IS NOT NULL
GROUP BY customer_id
UNION ALL
SELECT -1, 'UNKNOWN', NULL

In [ ]:
%%sql
DROP TABLE IF EXISTS insurance.dim_policy

In [ ]:
%%sql
CREATE TABLE insurance.dim_policy USING DELTA AS
SELECT xxhash64(policy_number) AS policy_key, policy_number, max(line_of_business) AS line_of_business
FROM (
  SELECT policy_number, CAST(NULL AS STRING) AS line_of_business FROM silver.insurance_claims_events
  UNION ALL
  SELECT policy_number, line_of_business FROM silver.insurance_policy_admin
) src
WHERE policy_number IS NOT NULL
GROUP BY policy_number
UNION ALL
SELECT -1, 'UNKNOWN', NULL

In [ ]:
%%sql
DROP TABLE IF EXISTS insurance.dim_location

In [ ]:
%%sql
CREATE TABLE insurance.dim_location USING DELTA AS
SELECT DISTINCT CASE WHEN coalesce(postal_code, state_code) IS NULL THEN -1 ELSE xxhash64(coalesce(CAST(postal_code AS STRING), ''), coalesce(CAST(state_code AS STRING), '')) END AS location_key, postal_code, state_code
FROM (
  SELECT postal_code, state_code FROM silver.insurance_claims_events
  UNION ALL
  SELECT CAST(NULL AS STRING) AS postal_code, state_code FROM silver.insurance_policy_admin
) src
WHERE coalesce(postal_code, state_code) IS NOT NULL
UNION ALL
SELECT -1, NULL, NULL

In [ ]:
%%sql
DROP TABLE IF EXISTS insurance.dim_date

In [ ]:
%%sql
CREATE TABLE insurance.dim_date USING DELTA AS
SELECT CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key, d AS calendar_date, year(d) AS year,
  quarter(d) AS quarter, month(d) AS month, date_format(d, 'MMMM') AS month_name,
  dayofmonth(d) AS day_of_month, date_format(d, 'EEEE') AS day_name, dayofweek(d) IN (1, 7) AS is_weekend
FROM (SELECT explode(sequence(lo, hi, INTERVAL 1 DAY)) AS d
  FROM (SELECT min(d) AS lo, max(d) AS hi FROM (
    SELECT CAST(loss_date AS DATE) AS d FROM silver.insurance_claims_events
    UNION ALL
    SELECT CAST(claim_reported_timestamp AS DATE) AS d FROM silver.insurance_claims_events
    UNION ALL
    SELECT CAST(policy_effective_date AS DATE) AS d FROM silver.insurance_policy_admin
    UNION ALL
    SELECT CAST(policy_expiration_date AS DATE) AS d FROM silver.insurance_policy_admin
  ) dates) bounds)
UNION ALL
SELECT -1, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL

In [ ]:
%%sql
DROP TABLE IF EXISTS insurance.fact_claims_events

In [ ]:
%%sql
CREATE TABLE insurance.fact_claims_events USING DELTA AS
SELECT
  CASE WHEN customer_id IS NULL THEN -1 ELSE xxhash64(customer_id) END AS customer_key,
  CASE WHEN policy_number IS NULL THEN -1 ELSE xxhash64(policy_number) END AS policy_key,
  CASE WHEN coalesce(postal_code, state_code) IS NULL THEN -1 ELSE xxhash64(coalesce(CAST(postal_code AS STRING), ''), coalesce(CAST(state_code AS STRING), '')) END AS location_key,
  coalesce(CAST(date_format(loss_date, 'yyyyMMdd') AS INT), -1) AS loss_date_key,
  coalesce(CAST(date_format(claim_reported_timestamp, 'yyyyMMdd') AS INT), -1) AS claim_reported_date_key,
  claim_id,
  claim_reported_timestamp,
  claim_status,
  loss_cause,
  reserve_amount,
  paid_amount,
  _source
FROM silver.insurance_claims_events

In [ ]:
%%sql
DROP TABLE IF EXISTS insurance.fact_policy_admin

In [ ]:
%%sql
CREATE TABLE insurance.fact_policy_admin USING DELTA AS
SELECT
  CASE WHEN agent_id IS NULL THEN -1 ELSE xxhash64(agent_id) END AS agent_key,
  CASE WHEN customer_id IS NULL THEN -1 ELSE xxhash64(customer_id) END AS customer_key,
  CASE WHEN policy_number IS NULL THEN -1 ELSE xxhash64(policy_number) END AS policy_key,
  CASE WHEN coalesce(state_code) IS NULL THEN -1 ELSE xxhash64('', coalesce(CAST(state_code AS STRING), '')) END AS location_key,
  coalesce(CAST(date_format(policy_effective_date, 'yyyyMMdd') AS INT), -1) AS policy_effective_date_key,
  coalesce(CAST(date_format(policy_expiration_date, 'yyyyMMdd') AS INT), -1) AS policy_expiration_date_key,
  premium_amount,
  _source
FROM silver.insurance_policy_admin

Validation for insurance: every count must be 0.

In [ ]:
%%sql
SELECT 'unique insurance.dim_agent.agent_key' AS check, (SELECT count(*) - count(DISTINCT agent_key) FROM insurance.dim_agent) AS violations
UNION ALL
SELECT 'unique insurance.dim_customer.customer_key' AS check, (SELECT count(*) - count(DISTINCT customer_key) FROM insurance.dim_customer) AS violations
UNION ALL
SELECT 'unique insurance.dim_policy.policy_key' AS check, (SELECT count(*) - count(DISTINCT policy_key) FROM insurance.dim_policy) AS violations
UNION ALL
SELECT 'unique insurance.dim_location.location_key' AS check, (SELECT count(*) - count(DISTINCT location_key) FROM insurance.dim_location) AS violations
UNION ALL
SELECT 'unique insurance.dim_date.date_key' AS check, (SELECT count(*) - count(DISTINCT date_key) FROM insurance.dim_date) AS violations
UNION ALL
SELECT 'rows insurance.fact_claims_events' AS check, (SELECT (SELECT count(*) FROM insurance.fact_claims_events) - (SELECT count(*) FROM silver.insurance_claims_events)) AS violations
UNION ALL
SELECT 'orphans insurance.fact_claims_events.customer_key' AS check, (SELECT count(*) FROM insurance.fact_claims_events f LEFT ANTI JOIN insurance.dim_customer d ON f.customer_key = d.customer_key) AS violations
UNION ALL
SELECT 'orphans insurance.fact_claims_events.policy_key' AS check, (SELECT count(*) FROM insurance.fact_claims_events f LEFT ANTI JOIN insurance.dim_policy d ON f.policy_key = d.policy_key) AS violations
UNION ALL
SELECT 'orphans insurance.fact_claims_events.location_key' AS check, (SELECT count(*) FROM insurance.fact_claims_events f LEFT ANTI JOIN insurance.dim_location d ON f.location_key = d.location_key) AS violations
UNION ALL
SELECT 'orphans insurance.fact_claims_events.loss_date_key' AS check, (SELECT count(*) FROM insurance.fact_claims_events f LEFT ANTI JOIN insurance.dim_date d ON f.loss_date_key = d.date_key) AS violations
UNION ALL
SELECT 'orphans insurance.fact_claims_events.claim_reported_date_key' AS check, (SELECT count(*) FROM insurance.fact_claims_events f LEFT ANTI JOIN insurance.dim_date d ON f.claim_reported_date_key = d.date_key) AS violations
UNION ALL
SELECT 'rows insurance.fact_policy_admin' AS check, (SELECT (SELECT count(*) FROM insurance.fact_policy_admin) - (SELECT count(*) FROM silver.insurance_policy_admin)) AS violations
UNION ALL
SELECT 'orphans insurance.fact_policy_admin.agent_key' AS check, (SELECT count(*) FROM insurance.fact_policy_admin f LEFT ANTI JOIN insurance.dim_agent d ON f.agent_key = d.agent_key) AS violations
UNION ALL
SELECT 'orphans insurance.fact_policy_admin.customer_key' AS check, (SELECT count(*) FROM insurance.fact_policy_admin f LEFT ANTI JOIN insurance.dim_customer d ON f.customer_key = d.customer_key) AS violations
UNION ALL
SELECT 'orphans insurance.fact_policy_admin.policy_key' AS check, (SELECT count(*) FROM insurance.fact_policy_admin f LEFT ANTI JOIN insurance.dim_policy d ON f.policy_key = d.policy_key) AS violations
UNION ALL
SELECT 'orphans insurance.fact_policy_admin.location_key' AS check, (SELECT count(*) FROM insurance.fact_policy_admin f LEFT ANTI JOIN insurance.dim_location d ON f.location_key = d.location_key) AS violations
UNION ALL
SELECT 'orphans insurance.fact_policy_admin.policy_effective_date_key' AS check, (SELECT count(*) FROM insurance.fact_policy_admin f LEFT ANTI JOIN insurance.dim_date d ON f.policy_effective_date_key = d.date_key) AS violations
UNION ALL
SELECT 'orphans insurance.fact_policy_admin.policy_expiration_date_key' AS check, (SELECT count(*) FROM insurance.fact_policy_admin f LEFT ANTI JOIN insurance.dim_date d ON f.policy_expiration_date_key = d.date_key) AS violations

## Retail

In [ ]:
%%sql
SET spark.sql.session.timeZone = UTC

In [ ]:
%%sql
CREATE SCHEMA IF NOT EXISTS bronze

In [ ]:
%%sql
CREATE SCHEMA IF NOT EXISTS silver

In [ ]:
%%sql
CREATE SCHEMA IF NOT EXISTS retail

In [ ]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW raw_retail_ecommerce_orders USING json OPTIONS (path 'Files/raw/retail/ecommerce_orders.json', multiLine 'true')

In [ ]:
%%sql
DROP TABLE IF EXISTS bronze.retail_ecommerce_orders

In [ ]:
%%sql
CREATE TABLE bronze.retail_ecommerce_orders USING DELTA AS
SELECT * FROM raw_retail_ecommerce_orders

In [ ]:
%%sql
DROP TABLE IF EXISTS silver.retail_ecommerce_orders

In [ ]:
%%sql
CREATE TABLE silver.retail_ecommerce_orders USING DELTA AS
SELECT
  CAST(`order_id` AS STRING) AS order_id,
  CAST(`customer`.`id` AS STRING) AS customer_id,
  CAST(`customer`.`email` AS STRING) AS customer_email,
  CAST(`customer`.`name` AS STRING) AS customer_name,
  CAST(`placed_at` AS TIMESTAMP) AS order_timestamp,
  CAST(`items_total` AS DECIMAL(18,2)) AS order_subtotal,
  CAST(`shipping_fee` AS DECIMAL(18,2)) AS shipping_fee,
  CAST(`ship_to`.`postal_code` AS STRING) AS postal_code,
  CAST(`ship_to`.`country` AS STRING) AS country_code,
  CAST(`channel` AS STRING) AS sales_channel,
  'retail/ecommerce_orders.json' AS _source
FROM bronze.retail_ecommerce_orders

In [ ]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW raw_retail_pos_sales USING csv OPTIONS (path 'Files/raw/retail/pos_sales.csv', header 'true')

In [ ]:
%%sql
DROP TABLE IF EXISTS bronze.retail_pos_sales

In [ ]:
%%sql
CREATE TABLE bronze.retail_pos_sales USING DELTA AS
SELECT * FROM raw_retail_pos_sales

In [ ]:
%%sql
DROP TABLE IF EXISTS silver.retail_pos_sales

In [ ]:
%%sql
CREATE TABLE silver.retail_pos_sales USING DELTA AS
SELECT
  CAST(`txn_id` AS STRING) AS transaction_id,
  CAST(`store_no` AS STRING) AS store_id,
  CAST(`sku` AS STRING) AS product_sku,
  CAST(`qty` AS INT) AS quantity,
  CAST(`unit_price` AS DECIMAL(18,2)) AS unit_price,
  CAST(`disc_amt` AS DECIMAL(18,2)) AS discount_amount,
  CAST(`txn_ts` AS TIMESTAMP) AS sale_timestamp,
  CAST(`cust_loyalty_id` AS STRING) AS customer_id,
  'retail/pos_sales.csv' AS _source
FROM bronze.retail_pos_sales

In [ ]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW raw_retail_product_catalog USING csv OPTIONS (path 'Files/raw/retail/product_catalog.csv', header 'true')

In [ ]:
%%sql
DROP TABLE IF EXISTS bronze.retail_product_catalog

In [ ]:
%%sql
CREATE TABLE bronze.retail_product_catalog USING DELTA AS
SELECT * FROM raw_retail_product_catalog

In [ ]:
%%sql
DROP TABLE IF EXISTS silver.retail_product_catalog

In [ ]:
%%sql
CREATE TABLE silver.retail_product_catalog USING DELTA AS
SELECT
  CAST(`product_sku` AS STRING) AS product_sku,
  CAST(`product_title` AS STRING) AS product_name,
  CAST(`category_lvl1` AS STRING) AS product_category,
  CAST(`brand_name` AS STRING) AS brand_name,
  CAST(`list_price` AS DECIMAL(18,2)) AS list_price,
  CAST(`launch_dt` AS DATE) AS product_launch_date,
  'retail/product_catalog.csv' AS _source
FROM bronze.retail_product_catalog

In [ ]:
%%sql
DROP TABLE IF EXISTS retail.dim_customer

In [ ]:
%%sql
CREATE TABLE retail.dim_customer USING DELTA AS
SELECT xxhash64(customer_id) AS customer_key, customer_id, max(customer_email) AS customer_email, max(customer_name) AS customer_name
FROM (
  SELECT customer_id, customer_email, customer_name FROM silver.retail_ecommerce_orders
  UNION ALL
  SELECT customer_id, CAST(NULL AS STRING) AS customer_email, CAST(NULL AS STRING) AS customer_name FROM silver.retail_pos_sales
) src
WHERE customer_id IS NOT NULL
GROUP BY customer_id
UNION ALL
SELECT -1, 'UNKNOWN', NULL, NULL

In [ ]:
%%sql
DROP TABLE IF EXISTS retail.dim_product

In [ ]:
%%sql
CREATE TABLE retail.dim_product USING DELTA AS
SELECT xxhash64(product_sku) AS product_key, product_sku, max(brand_name) AS brand_name, max(list_price) AS list_price, max(product_category) AS product_category, max(product_name) AS product_name, max(product_launch_date) AS product_launch_date
FROM (
  SELECT product_sku, CAST(NULL AS STRING) AS brand_name, CAST(NULL AS DECIMAL(18,2)) AS list_price, CAST(NULL AS STRING) AS product_category, CAST(NULL AS STRING) AS product_name, CAST(NULL AS DATE) AS product_launch_date FROM silver.retail_pos_sales
  UNION ALL
  SELECT product_sku, brand_name, list_price, product_category, product_name, product_launch_date FROM silver.retail_product_catalog
) src
WHERE product_sku IS NOT NULL
GROUP BY product_sku
UNION ALL
SELECT -1, 'UNKNOWN', NULL, NULL, NULL, NULL, NULL

In [ ]:
%%sql
DROP TABLE IF EXISTS retail.dim_store

In [ ]:
%%sql
CREATE TABLE retail.dim_store USING DELTA AS
SELECT xxhash64(store_id) AS store_key, store_id
FROM (
  SELECT store_id FROM silver.retail_pos_sales
) src
WHERE store_id IS NOT NULL
GROUP BY store_id
UNION ALL
SELECT -1, 'UNKNOWN'

In [ ]:
%%sql
DROP TABLE IF EXISTS retail.dim_location

In [ ]:
%%sql
CREATE TABLE retail.dim_location USING DELTA AS
SELECT DISTINCT CASE WHEN coalesce(postal_code, country_code) IS NULL THEN -1 ELSE xxhash64(coalesce(CAST(postal_code AS STRING), ''), coalesce(CAST(country_code AS STRING), '')) END AS location_key, postal_code, country_code
FROM (
  SELECT postal_code, country_code FROM silver.retail_ecommerce_orders
) src
WHERE coalesce(postal_code, country_code) IS NOT NULL
UNION ALL
SELECT -1, NULL, NULL

In [ ]:
%%sql
DROP TABLE IF EXISTS retail.dim_date

In [ ]:
%%sql
CREATE TABLE retail.dim_date USING DELTA AS
SELECT CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key, d AS calendar_date, year(d) AS year,
  quarter(d) AS quarter, month(d) AS month, date_format(d, 'MMMM') AS month_name,
  dayofmonth(d) AS day_of_month, date_format(d, 'EEEE') AS day_name, dayofweek(d) IN (1, 7) AS is_weekend
FROM (SELECT explode(sequence(lo, hi, INTERVAL 1 DAY)) AS d
  FROM (SELECT min(d) AS lo, max(d) AS hi FROM (
    SELECT CAST(order_timestamp AS DATE) AS d FROM silver.retail_ecommerce_orders
    UNION ALL
    SELECT CAST(sale_timestamp AS DATE) AS d FROM silver.retail_pos_sales
  ) dates) bounds)
UNION ALL
SELECT -1, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL

In [ ]:
%%sql
DROP TABLE IF EXISTS retail.fact_ecommerce_orders

In [ ]:
%%sql
CREATE TABLE retail.fact_ecommerce_orders USING DELTA AS
SELECT
  CASE WHEN customer_id IS NULL THEN -1 ELSE xxhash64(customer_id) END AS customer_key,
  CASE WHEN coalesce(postal_code, country_code) IS NULL THEN -1 ELSE xxhash64(coalesce(CAST(postal_code AS STRING), ''), coalesce(CAST(country_code AS STRING), '')) END AS location_key,
  coalesce(CAST(date_format(order_timestamp, 'yyyyMMdd') AS INT), -1) AS order_date_key,
  order_id,
  order_timestamp,
  sales_channel,
  order_subtotal,
  shipping_fee,
  _source
FROM silver.retail_ecommerce_orders

In [ ]:
%%sql
DROP TABLE IF EXISTS retail.fact_pos_sales

In [ ]:
%%sql
CREATE TABLE retail.fact_pos_sales USING DELTA AS
SELECT
  CASE WHEN customer_id IS NULL THEN -1 ELSE xxhash64(customer_id) END AS customer_key,
  CASE WHEN product_sku IS NULL THEN -1 ELSE xxhash64(product_sku) END AS product_key,
  CASE WHEN store_id IS NULL THEN -1 ELSE xxhash64(store_id) END AS store_key,
  coalesce(CAST(date_format(sale_timestamp, 'yyyyMMdd') AS INT), -1) AS sale_date_key,
  transaction_id,
  sale_timestamp,
  quantity,
  unit_price,
  discount_amount,
  _source
FROM silver.retail_pos_sales

Validation for retail: every count must be 0.

In [ ]:
%%sql
SELECT 'unique retail.dim_customer.customer_key' AS check, (SELECT count(*) - count(DISTINCT customer_key) FROM retail.dim_customer) AS violations
UNION ALL
SELECT 'unique retail.dim_product.product_key' AS check, (SELECT count(*) - count(DISTINCT product_key) FROM retail.dim_product) AS violations
UNION ALL
SELECT 'unique retail.dim_store.store_key' AS check, (SELECT count(*) - count(DISTINCT store_key) FROM retail.dim_store) AS violations
UNION ALL
SELECT 'unique retail.dim_location.location_key' AS check, (SELECT count(*) - count(DISTINCT location_key) FROM retail.dim_location) AS violations
UNION ALL
SELECT 'unique retail.dim_date.date_key' AS check, (SELECT count(*) - count(DISTINCT date_key) FROM retail.dim_date) AS violations
UNION ALL
SELECT 'rows retail.fact_ecommerce_orders' AS check, (SELECT (SELECT count(*) FROM retail.fact_ecommerce_orders) - (SELECT count(*) FROM silver.retail_ecommerce_orders)) AS violations
UNION ALL
SELECT 'orphans retail.fact_ecommerce_orders.customer_key' AS check, (SELECT count(*) FROM retail.fact_ecommerce_orders f LEFT ANTI JOIN retail.dim_customer d ON f.customer_key = d.customer_key) AS violations
UNION ALL
SELECT 'orphans retail.fact_ecommerce_orders.location_key' AS check, (SELECT count(*) FROM retail.fact_ecommerce_orders f LEFT ANTI JOIN retail.dim_location d ON f.location_key = d.location_key) AS violations
UNION ALL
SELECT 'orphans retail.fact_ecommerce_orders.order_date_key' AS check, (SELECT count(*) FROM retail.fact_ecommerce_orders f LEFT ANTI JOIN retail.dim_date d ON f.order_date_key = d.date_key) AS violations
UNION ALL
SELECT 'rows retail.fact_pos_sales' AS check, (SELECT (SELECT count(*) FROM retail.fact_pos_sales) - (SELECT count(*) FROM silver.retail_pos_sales)) AS violations
UNION ALL
SELECT 'orphans retail.fact_pos_sales.customer_key' AS check, (SELECT count(*) FROM retail.fact_pos_sales f LEFT ANTI JOIN retail.dim_customer d ON f.customer_key = d.customer_key) AS violations
UNION ALL
SELECT 'orphans retail.fact_pos_sales.product_key' AS check, (SELECT count(*) FROM retail.fact_pos_sales f LEFT ANTI JOIN retail.dim_product d ON f.product_key = d.product_key) AS violations
UNION ALL
SELECT 'orphans retail.fact_pos_sales.store_key' AS check, (SELECT count(*) FROM retail.fact_pos_sales f LEFT ANTI JOIN retail.dim_store d ON f.store_key = d.store_key) AS violations
UNION ALL
SELECT 'orphans retail.fact_pos_sales.sale_date_key' AS check, (SELECT count(*) FROM retail.fact_pos_sales f LEFT ANTI JOIN retail.dim_date d ON f.sale_date_key = d.date_key) AS violations